In [1]:
import time
import requests
from pathlib import Path
import pandas as pd

import mygene
from brainimagelibrary import retrieve, query

In [2]:
df_path = Path('./data/gene_datasets - Gene × Dataset Matrix.csv')

In [3]:
df = pd.read_csv(df_path, index_col=0)

In [4]:
# change df to boolean
df = ~df.isna()
df.head()

,ace-ban-oat,ace-can-ill,ace-can-imp,ace-can-ink,ace-dry-dip,ace-dry-dog,ace-dry-dry,ace-dry-dub,ace-dud-vex,ace-dud-vow,...,ace-low-cap,ace-low-car,ace-low-cat,ace-low-cop,ace-low-cot,ace-low-cow,ace-low-cry,ace-low-cub,ace-low-zip,ace-met-war
Gene,,,,,,,,,,,,,,,,,,,,,
1500015O10Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1700011I03Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2900052N01Rik,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
4930509J09Rik,False,False,False,False,False,False,False,False,False,False,...,True,True,True,True,True,True,True,True,True,False
9030622O22Rik,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Get species taxids

In [5]:
# get species names from the metadata for each dataset

species_list = []
for i in range(len(df.columns)):
    metadata = retrieve.by_id(bildid=df.columns[i])
    species = metadata['retjson'][0]['Specimen'][0]['species']
    species_list.append(species)

species_list_unique = list(set(species_list))
species_list_unique 

['Mouse', 'human', 'mouse', 'Human', 'Rhesus Macaque']

In [6]:
def common_name_to_taxid(name):
    """Returns (taxid, scientific_name), or (None, None) if not found."""
    # Step 1: search by name to get taxid
    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
        params={"db": "taxonomy", "term": name, "retmode": "json"},
    )
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    if not ids:
        return None
    return int(ids[0])

def taxids_to_names(taxids):
    taxids = list(set(str(t) for t in taxids))  # dedupe
    r = requests.get(
        "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
        params={"db": "taxonomy", "id": ",".join(taxids), "retmode": "json"},
    )
    result = r.json().get("result", {})
    return {int(tid): result.get(tid, {}).get("scientificname") for tid in taxids}

In [7]:
# from metadata entry to taxid
taxid_map = {}
for common_name in species_list_unique:
    taxid = common_name_to_taxid(common_name)

    taxid_map[common_name] = taxid

    time.sleep(0.33)
    
# from taxid to scientific name
name_map = taxids_to_names(list(taxid_map.values()))

for common_name, taxid in taxid_map.items():
    print(f"{common_name} -> {taxid} -> {name_map.get(taxid)}")

Mouse -> 10090 -> Mus musculus
human -> 9606 -> Homo sapiens
mouse -> 10090 -> Mus musculus
Human -> 9606 -> Homo sapiens
Rhesus Macaque -> 9544 -> Macaca mulatta


## Resolve gene names

In [8]:
# find taxids for each column
def resolve(col):
    metadata = retrieve.by_id(bildid=col)
    species = metadata['retjson'][0]['Specimen'][0]['species']
    taxid = taxid_map.get(species)
    return taxid

col_taxid = {col: resolve(col) for col in df.columns}

In [9]:
# get genes for each taxid
taxid_to_genes: dict[int, set[str]] = {}
for col, taxid in col_taxid.items():
    present = df.index[df[col]]
    taxid_to_genes.setdefault(taxid, set()).update(present)

In [10]:
len(taxid_to_genes)

3

In [13]:
from collections import Counter
from enum import Enum
from typing import Iterable

import mygene

mg = mygene.MyGeneInfo()

class MatchStatus(str, Enum):
    SYMBOL_RESOLVED = "symbol_resolved"  # 1 hit via strict symbol scope
    SYMBOL_AMB      = "symbol_amb"       # >1 hit via strict symbol scope
    ALIAS_RESOLVED  = "alias_resolved"   # 1 hit via alias/xref scopes (symbol-pass missed)
    ALIAS_AMB       = "alias_amb"        # >1 hit via alias/xref scopes
    NONE            = "none"             # no hit in either pass


def classify_for_species(
    genes: Iterable[str],
    taxid: int,
    mg: mygene.MyGeneInfo,
) -> dict[str, MatchStatus]:
    """Two-pass query, tracking both match type (symbol vs alias) and
    ambiguity (single hit vs multiple hits)."""
    genes = sorted(set(genes))
    if not genes:
        return {}

    # Pass 1 — exact symbol only
    r1 = mg.querymany(
        genes, scopes="symbol", fields="symbol",
        species=taxid, returnall=True,
    )
    sym_counts = Counter(h["query"] for h in r1["out"] if not h.get("notfound"))
    misses = list(r1["missing"])

    # Pass 2 — alias / xref scopes, only on what didn't resolve as a symbol
    alias_counts: Counter = Counter()
    if misses:
        r2 = mg.querymany(
            misses, scopes="alias,ensembl.gene,entrezgene,name", fields="symbol",
            species=taxid, returnall=True,
        )
        alias_counts = Counter(h["query"] for h in r2["out"] if not h.get("notfound"))

    def status(g: str) -> MatchStatus:
        if g in sym_counts:
            return MatchStatus.SYMBOL_RESOLVED if sym_counts[g] == 1 else MatchStatus.SYMBOL_AMB
        if g in alias_counts:
            return MatchStatus.ALIAS_RESOLVED if alias_counts[g] == 1 else MatchStatus.ALIAS_AMB
        return MatchStatus.NONE

    return {g: status(g) for g in genes}

In [14]:
status_by_taxid = {
    taxid: pd.Series(
        {g: s.value for g, s in classify_for_species(genes, taxid, mg).items()}
    )
    for taxid, genes in taxid_to_genes.items()
}

1 input query terms found dup hits:	[('B230209E15Rik', 2)]
48 input query terms found no hit:	['1500015O10Rik', '1700011I03Rik', 'Cars', 'Fyb', 'Hist1h1d', 'Lhfp', 'Prrxl1', 'Sept4', 'blank-0', 
2 input query terms found dup hits:	[('Lhfp', 2), ('volume', 10)]
38 input query terms found no hit:	['blank-0', 'blank-1', 'blank-10', 'blank-11', 'blank-12', 'blank-13', 'blank-14', 'blank-15', 'blan
10 input query terms found dup hits:	[('CASC6', 2), ('DLX6-AS1', 2), ('IGHA1', 2), ('LINC01965', 2), ('LINC02822', 2), ('NR2F2-AS1', 2), 
5 input query terms found no hit:	['FAM155A', 'FAM160A1', 'FAM189A2', 'NDUFA4L2', 'TCTEX1D1']
4 input query terms found dup hits:	[('ADCYAP1', 2), ('CLDN5', 2), ('DPF3', 2), ('TSHZ2', 2)]
1 input query terms found no hit:	['ZNF429']


In [59]:
status_by_taxid

{10090: 1500015O10Rik            alias_resolved
 1700011I03Rik            alias_resolved
 2900052N01Rik           symbol_resolved
 4930509J09Rik           symbol_resolved
 9030622O22Rik           symbol_resolved
                              ...       
 global_y                           none
 mRNA_counts                        none
 median_total_density               none
 polyT                              none
 volume                        alias_amb
 Length: 1184, dtype: str,
 9606: ABCA8      symbol_resolved
 ABCB1      symbol_resolved
 ABCG2      symbol_resolved
 ABLIM1     symbol_resolved
 ACKR3      symbol_resolved
                 ...       
 ZFPM2      symbol_resolved
 ZIC1       symbol_resolved
 ZNF385D    symbol_resolved
 ZNF536     symbol_resolved
 ZNF804B    symbol_resolved
 Length: 1005, dtype: str,
 9544: ABO         symbol_resolved
 ADAM12      symbol_resolved
 ADAMTS18    symbol_resolved
 ADAMTSL1    symbol_resolved
 ADARB2      symbol_resolved
                  ...  

In [15]:
# Broadcast back: for each column, look up its species' status table,
# then mask out genes that aren't actually present in that dataset.
out = pd.DataFrame(index=df.index, columns=df.columns, dtype=object)
for col, taxid in col_taxid.items():
    s = status_by_taxid[taxid].reindex(df.index)
    out[col] = s.where(df[col].fillna(False).astype(bool))

In [16]:
out

,ace-ban-oat,ace-can-ill,ace-can-imp,ace-can-ink,ace-dry-dip,ace-dry-dog,ace-dry-dry,ace-dry-dub,ace-dud-vex,ace-dud-vow,...,ace-low-cap,ace-low-car,ace-low-cat,ace-low-cop,ace-low-cot,ace-low-cow,ace-low-cry,ace-low-cub,ace-low-zip,ace-met-war
Gene,,,,,,,,,,,,,,,,,,,,,
1500015O10Rik,alias_resolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1700011I03Rik,alias_resolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2900052N01Rik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,NaN
4930509J09Rik,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,symbol_resolved,NaN
9030622O22Rik,symbol_resolved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
global_y,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mRNA_counts,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
median_total_density,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
summary = (
    out.apply(lambda c: c.value_counts())
             .reindex(["symbol_resolved", "symbol_amb", "alias_resolved", "alias_amb", "none"])
             .fillna(0).astype(int)
             .T   # datasets as rows, statuses as columns
)

In [22]:
summary

,symbol_resolved,symbol_amb,alias_resolved,alias_amb,none
ace-ban-oat,285,0,4,1,38
ace-can-ill,294,4,2,0,0
ace-can-imp,294,4,2,0,0
ace-can-ink,294,4,2,0,0
ace-dry-dip,294,4,2,0,0
...,...,...,...,...,...
ace-low-cow,497,0,2,1,0
ace-low-cry,497,0,2,1,0
ace-low-cub,497,0,2,1,0
ace-low-zip,497,0,2,1,0


## Manually look at the unresolved

In [31]:
# get genes for each taxid
taxid_to_unresolved: dict[int, set[str]] = {}
for col, taxid in col_taxid.items():
    present = out.index[(out[col]=='symbol_amb') | (out[col]=='alias_amb') ]
    taxid_to_unresolved.setdefault(taxid, set()).update(present)

In [32]:
taxid_to_unresolved

{10090: {'B230209E15Rik', 'Lhfp', 'volume'},
 9606: {'CASC6',
  'DLX6-AS1',
  'IGHA1',
  'LINC01965',
  'LINC02822',
  'NR2F2-AS1',
  'SATB1-AS1',
  'TRBC1',
  'TRDC',
  'XACT'},
 9544: {'ADCYAP1', 'CLDN5', 'DPF3', 'TSHZ2'}}

In [40]:
def is_soft_duplicate(hits: list[dict]) -> bool:
    """Detects mygene's NCBI/Ensembl unmerged-record artifact."""
    if len(hits) < 2:
        return False
    # Same symbol, name, taxid across all hits
    keys = {(h.get("symbol"), h.get("name"), h.get("taxid")) for h in hits}
    if len(keys) != 1:
        return False
    # Each hit carries data from only one of {ensembl, entrezgene}, no overlap
    has_ens = [h for h in hits if h.get("ensembl")]
    has_ent = [h for h in hits if h.get("entrezgene")]
    return len(has_ens) >= 1 and len(has_ent) >= 1 and not any(
        h.get("ensembl") and h.get("entrezgene") for h in hits
    )

In [48]:
taxid = 9606  # human   
gene_list = list(taxid_to_unresolved[taxid])

results = mg.querymany(
    gene_list,
    scopes="symbol,alias,name,ensembl.gene,entrezgene",
    fields="symbol,name,taxid,entrezgene,ensembl.gene,alias",
    species=taxid,
    returnall=True,
)

for gene in gene_list:
    hits = [h for h in results["out"] if h["query"] == gene and not h.get("notfound")]
    if is_soft_duplicate(hits):
        print(f"{gene} has {len(hits)} hits but looks like a soft duplicate:")
        for h in hits:
            print(f"  - {h}")
    else:
        print(f"{gene} has {len(hits)} hits")

10 input query terms found dup hits:	[('TRBC1', 2), ('IGHA1', 2), ('LINC02822', 2), ('CASC6', 2), ('LINC01965', 2), ('SATB1-AS1', 2), ('T


TRBC1 has 2 hits
IGHA1 has 2 hits
LINC02822 has 2 hits but looks like a soft duplicate:
  - {'query': 'LINC02822', '_id': 'ENSG00000286021', '_score': 26.67398, 'ensembl': {'gene': 'ENSG00000286021'}, 'name': 'long intergenic non-protein coding RNA 2822', 'symbol': 'LINC02822', 'taxid': 9606}
  - {'query': 'LINC02822', '_id': '102724834', '_score': 26.67398, 'entrezgene': '102724834', 'name': 'long intergenic non-protein coding RNA 2822', 'symbol': 'LINC02822', 'taxid': 9606}
CASC6 has 2 hits but looks like a soft duplicate:
  - {'query': 'CASC6', '_id': 'ENSG00000224944', '_score': 26.672888, 'ensembl': {'gene': 'ENSG00000224944'}, 'name': 'cancer susceptibility 6', 'symbol': 'CASC6', 'taxid': 9606}
  - {'query': 'CASC6', '_id': '101929083', '_score': 26.672888, 'entrezgene': '101929083', 'name': 'cancer susceptibility 6', 'symbol': 'CASC6', 'taxid': 9606}
LINC01965 has 2 hits but looks like a soft duplicate:
  - {'query': 'LINC01965', '_id': 'ENSG00000256637', '_score': 26.674496, 'e

Comment: TRBC1 and IGHA1 sit in immunoreceptor loci (T-cell receptor β and immunoglobulin heavy chain respectively), the most haplotype-rich regions in the human genome. Ensembl assembles those loci on multiple alt contigs because the diversity is too high to collapse into a single reference. 

In [49]:
taxid = 10090  # mouse
gene_list = list(taxid_to_unresolved[taxid])

results = mg.querymany(
    gene_list,
    scopes="symbol,alias,name,ensembl.gene,entrezgene",
    fields="symbol,name,taxid,entrezgene,ensembl.gene,alias",
    species=taxid,
    returnall=True,
)

for gene in gene_list:
    hits = [h for h in results["out"] if h["query"] == gene and not h.get("notfound")]
    if is_soft_duplicate(hits):
        print(f"{gene} has {len(hits)} hits but looks like a soft duplicate:")
        for h in hits:
            print(f"  - {h}")
    else:
        print(f"{gene} has {len(hits)} hits")

3 input query terms found dup hits:	[('volume', 10), ('B230209E15Rik', 2), ('Lhfp', 2)]


volume has 10 hits
B230209E15Rik has 2 hits but looks like a soft duplicate:
  - {'query': 'B230209E15Rik', '_id': 'ENSMUSG00000109006', '_score': 22.372814, 'ensembl': {'gene': 'ENSMUSG00000109006'}, 'name': 'RIKEN cDNA B230209E15 gene', 'symbol': 'B230209E15Rik', 'taxid': 10090}
  - {'query': 'B230209E15Rik', '_id': '319752', '_score': 22.372814, 'entrezgene': '319752', 'name': 'RIKEN cDNA B230209E15 gene', 'symbol': 'B230209E15Rik', 'taxid': 10090}
Lhfp has 2 hits


Lhfp seems to be a trully ambiguous - two different members from the same family - Lhfpl1 and Lhfpl6.

In [58]:
# check other artifacts of these datasets - do they include probe sequences or probe IDs 

In [51]:
taxid = 9544  # macaque
gene_list = list(taxid_to_unresolved[taxid])

results = mg.querymany(
    gene_list,
    scopes="symbol,alias,name,ensembl.gene,entrezgene",
    fields="symbol,name,taxid,entrezgene,ensembl.gene,alias",
    species=taxid,
    returnall=True,
)

for gene in gene_list:
    hits = [h for h in results["out"] if h["query"] == gene and not h.get("notfound")]
    if is_soft_duplicate(hits):
        print(f"{gene} has {len(hits)} hits but looks like a soft duplicate:")
        for h in hits:
            print(f"  - {h}")
    else:
        print(f"{gene} has {len(hits)} hits")

4 input query terms found dup hits:	[('ADCYAP1', 2), ('DPF3', 2), ('TSHZ2', 2), ('CLDN5', 2)]


ADCYAP1 has 2 hits but looks like a soft duplicate:
  - {'query': 'ADCYAP1', '_id': 'ENSMMUG00000062768', '_score': 11.759855, 'ensembl': {'gene': 'ENSMMUG00000062768'}, 'name': 'adenylate cyclase activating polypeptide 1', 'symbol': 'ADCYAP1', 'taxid': 9544}
  - {'query': 'ADCYAP1', '_id': '100424113', '_score': 11.759855, 'alias': 'PACAP', 'entrezgene': '100424113', 'name': 'adenylate cyclase activating polypeptide 1', 'symbol': 'ADCYAP1', 'taxid': 9544}
DPF3 has 2 hits but looks like a soft duplicate:
  - {'query': 'DPF3', '_id': '714778', '_score': 11.372852, 'entrezgene': '714778', 'name': 'double PHD fingers 3', 'symbol': 'DPF3', 'taxid': 9544}
  - {'query': 'DPF3', '_id': 'ENSMMUG00000016351', '_score': 11.372852, 'ensembl': {'gene': 'ENSMMUG00000016351'}, 'name': 'double PHD fingers 3', 'symbol': 'DPF3', 'taxid': 9544}
TSHZ2 has 2 hits but looks like a soft duplicate:
  - {'query': 'TSHZ2', '_id': 'ENSMMUG00000013035', '_score': 11.50138, 'ensembl': {'gene': 'ENSMMUG00000013035